# Hướng dẫn Chạy mô hình Multimodal (Late Fusion) trên Google Colab
Notebook này đã được cấu hình tự động 100% để bạn có thể chạy từ đầu đến cuối mà không gặp lỗi.

**LƯU Ý TRƯỚC KHI CHẠY:**
Bạn cần đảm bảo đã tạo thư mục Lối tắt (Shortcut) tên là `SE365` trên Google Drive của bạn (trỏ từ link chia sẻ dữ liệu của nhóm). Nếu bạn đặt tên lối tắt khác, hãy sửa tên thư mục ở **Bước 3** và **Bước 3.5** 

### BƯỚC 1: Mount Google Drive
Lệnh này sẽ yêu cầu bạn cấp quyền truy cập Google Drive. Chúng ta cần làm điều này để đọc dữ liệu gốc (5000 ảnh và CSV) thông qua Lối tắt (Shortcut) mà không cần tải lại file zip.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### BƯỚC 2: Clone mã nguồn và cài đặt thư viện
Tải phiên bản code mới nhất từ Github và cài đặt các thư viện cần thiết (PyTorch, Transformers, Timm, ...).


In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt

### BƯỚC 3: Nạp dữ liệu vào máy ảo Colab bằng gdown (QUAN TRỌNG)
Chúng ta sẽ tải trực tiếp file nén `data.zip` từ Google Drive thông qua gdown. Bạn cần upload `data.zip` lên Drive, chuột phải chọn Share -> Anyone with the link, sau đó copy File ID (đoạn mã dài trên URL).

**Chỗ cần sửa:** Dán File ID của bạn thay cho chuỗi `YOUR_FILE_ID` bên dưới.

In [ ]:
!rm -rf ./data
# SỬA LẠI FILE ID NẾU CẦN
!gdown 11WoeUn2visKtGN5oOX9c2I6Grz3P88vD -O data.zip

# Giải nén data.zip (mặc định zip không làm giảm chất lượng ảnh)
!unzip -q data.zip

# Xoá file nén sau khi giải xong để nhẹ ổ cứng
!rm data.zip

# Kiểm tra xem dữ liệu đã được nạp chưa
!ls -la ./data

### BƯỚC 3.5: Cấu hình nơi lưu Checkpoint (Trọng số mô hình)
Để tránh bị mất kết quả khi Colab tự ngắt, đoạn code sau sẽ tạo một thư mục con trong `checkpoints/` trên Drive của bạn, với tên là thời gian hiện tại (Ví dụ: `20260611_083908`). 
Tất cả các mô hình Text, Image, Fusion sau khi train xong sẽ được tự động copy vào chung thư mục này.

**Chỗ cần sửa:** Nếu bạn muốn lưu vào thư mục khác, hãy sửa biến `drive_ckpt_path`.


In [ ]:
# BƯỚC 3.5: Khởi tạo thư mục lưu trữ cho toàn bộ phiên chạy này
# Đảm bảo tất cả mô hình train trong hôm nay đều nằm chung một thư mục
import os
import datetime

run_id = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
drive_ckpt_path = f'/content/drive/MyDrive/SE365/checkpoints/{run_id}'
os.environ['DRIVE_CKPT'] = drive_ckpt_path

# Tạo thư mục con checkpoints và thư mục run_id bằng os.makedirs
os.makedirs(drive_ckpt_path, exist_ok=True)
print(f'Mọi checkpoint trong phiên này sẽ được lưu chung vào: {drive_ckpt_path}')

### BƯỚC 4: Huấn luyện mô hình Text (XLM-RoBERTa)
**Các tham số bạn có thể tuỳ chỉnh ở lệnh Train (cả 3 mô hình text, imame, fusion):**

Các tham số liên quan đường dẫn
- `--train_path`: Đường dẫn đến file train.csv *(Mặc định: `./data/text/train.csv`)*
- `--val_path`: Đường dẫn đến file val.csv *(Mặc định: `./data/text/val.csv`)*
- `--test_path`: Đường dẫn đến file test.csv *(Mặc định: `./data/text/test.csv`)*
- `--image_dir`: Đường dẫn đến thư mục ảnh *(Mặc định: `./data/image`)*

Các tham số liên quan huấn luyện mô hình
- `--mode`: Chế độ chạy (`train_text`, `train_image`, `train_fusion`)
- `--epochs`: Số vòng lặp (Mặc định: 5)
- `--batch_size`: Kích thước batch (Mặc định: 16)
- `--lr`: Learning rate (Mặc định: 2e-5)
- `--alpha`: Trọng số loss cho các yếu tố phụ (Mặc định: 0.5)

In [ ]:
# BƯỚC 4: Huấn luyện mô hình Text (XLM-RoBERTa)
!python main.py --mode train_text --epochs 15 --grad_accum_steps 4

# Lưu Checkpoint Text ngay lập tức
!cp ./checkpoints/best_model_train_text.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Text vào $DRIVE_CKPT"

### BƯỚC 5: Huấn luyện mô hình Image (ConvNeXt)

In [ ]:
# BƯỚC 5: Huấn luyện mô hình Image (ConvNeXt)
!python main.py --mode train_image --epochs 15 --grad_accum_steps 4

# Lưu Checkpoint Image ngay lập tức
!cp ./checkpoints/best_model_train_image.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Image vào $DRIVE_CKPT"

### BƯỚC 6: Huấn luyện mô hình Fusion (Kết hợp Text + Image)
Mô hình này sẽ tự động tải các checkpoint tốt nhất của mô hình Text và Image vừa train ở trên để tiếp tục huấn luyện phần kết hợp.

In [ ]:
# BƯỚC 6: Huấn luyện mô hình Fusion (Kết hợp Text + Image)
!python main.py --mode train_fusion --epochs 10 --grad_accum_steps 4 --unfreeze_text_layers 1 --unfreeze_image_layers 1

# Lưu Checkpoint Fusion ngay lập tức
!cp ./checkpoints/best_model_train_fusion.pth $DRIVE_CKPT/ && echo "Đã lưu checkpoint Fusion vào $DRIVE_CKPT"

### BƯỚC 7: Đánh giá (Test) trên mô hình tốt nhất
Bước cuối cùng là chạy đánh giá mô hình Fusion trên tập dữ liệu Test chưa từng được thấy để tính ra các chỉ số sai số MAE, MSE và RMSE.

**Các tham số bạn có thể tuỳ chỉnh ở lệnh Test:** Khác với Train, lệnh Test chỉ quan tâm tới các tham số: `--test_path`, `--image_dir` và `--batch_size`.

In [ ]:
# BƯỚC 7: Đánh giá mô hình (Tính các metric MAE, MSE, RMSE)
!python test.py --mode train_fusion